## Initialize Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Clusters_amritap1")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.memory", "10g")
    .config("spark.executor.cores", "1")
    .config("spark.executor.instances", "14")
    .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer","1g")
    .config("spark.driver.maxResultSize","8g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/12 06:29:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/12 06:29:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Load Raw Data

In [2]:
commentsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/comments")

In [3]:
submissionsDF = spark.read.parquet("/project/macs40123/amritap1/macs-40123-amritapathak1/final_data_with_id/submissions")

In [4]:
commentsDF.printSchema()

root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- body: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- score: long (nullable = true)



In [5]:
submissionsDF.printSchema()

root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- title: string (nullable = true)



## Data Cleaning

I implemented a two-step cleaning process:
1. **Remove bot accounts** - Filter out automoderator, bot, and moderator accounts
2. **Remove deleted content** - Drop `[deleted]` and `[removed]` posts

Keeping bots would artificially inflate certain ticker associations (e.g., automated stock updates), making the patterns less meaningful for understanding organic user discussions.

In [6]:
from pyspark.sql import functions as F

def remove_bot_accounts(df):
    a = F.lower(F.col("author"))
    is_bot = F.coalesce(
        a.contains("automoderator") | a.contains("moderator") | a.contains("bot") | a.contains("automod"),
        F.lit(False)
    )
    df = df.filter(~is_bot)
    return df

In [7]:
commentsDF = remove_bot_accounts(commentsDF)
submissionsDF = remove_bot_accounts(submissionsDF)

In [8]:
from pyspark.sql import functions as F

def remove_deleted_content(df, text_column):
    df = df.filter(
        F.col(text_column).isNotNull() &
        ~F.col(text_column).isin("[deleted]", "[removed]")
    )
    return df

In [9]:
commentsDF = remove_deleted_content(commentsDF, 'body')
submissionsDF = remove_deleted_content(submissionsDF, 'selftext')

## Data Preparation

- **Comments**: Extract `id`, mark source as "comment", use `body` as text
- **Submissions**: Extract `id`, mark source as "submission", concatenate `title` and `selftext`

In [10]:
comments = commentsDF.select(
    F.col("id").alias("id"),
    F.lit("comment").alias("src"),
    F.col("body").alias("text")
).where(F.col("text").isNotNull())

submissions = submissionsDF.select(
    F.col("id").alias("id"),
    F.lit("submission").alias("src"),
    F.concat_ws(" ", F.coalesce("title", F.lit("")), F.coalesce("selftext", F.lit(""))).alias("text")
).where(F.col("text").isNotNull())

In [11]:
comments.printSchema()

root
 |-- id: string (nullable = true)
 |-- src: string (nullable = false)
 |-- text: string (nullable = true)



In [12]:
submissions.printSchema()

root
 |-- id: string (nullable = true)
 |-- src: string (nullable = false)
 |-- text: string (nullable = false)



In [14]:
comments = comments.cache()

25/12/12 06:30:46 WARN CacheManager: Asked to cache already cached data.


In [15]:
submissions = submissions.cache()

# Mining Frequent Itemsets

# Frequent Itemset Analysis (2020–2022)

This notebook applies frequent itemset mining to Reddit discussion data from **2020–2022**, a period characterized by heightened retail trading activity, pandemic-era uncertainty, and episodic collective coordination. The analysis focuses on identifying recurring co-occurrence patterns in how users discuss assets, emotions, and trading behaviors.


In [13]:
from pyspark.sql import functions as F, types as T

tickers = ["AAPL","TSLA","NVDA","GME","AMC","MSFT"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_simple_tickers(text: str):
    if not text:
        return []
    s = text.upper()
    found = [t for t in ticker_set if t in s]
    return [f"T_{t}" for t in found]

In [14]:
submissions = submissions.select(
    "id",
    extract_simple_tickers(F.col("text")).alias("tickers")
)

In [15]:
comments = comments.select(
    "id",
    extract_simple_tickers("text").alias("tickers")
)

## Creating Buckets

In [16]:
comment_groups = (comments
    .groupBy("id")
    .agg(F.array_distinct(F.flatten(F.collect_list("tickers"))).alias("c_tickers")))

In [17]:
baskets = (submissions.alias("s")
    .join(comment_groups.alias("c"), F.col("s.id") == F.col("c.id"), "left")
    .select(
        F.col("s.id"),
        F.array_distinct(F.array_union(F.coalesce(F.col("s.tickers"), F.array()),
                                       F.coalesce(F.col("c.c_tickers"), F.array()))).alias("items"))
    .filter(F.size("items") >= 2))

## FP-Growth

### Limited Ticker Set

In [18]:
from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.1).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))

In [19]:
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))

In [20]:
pairs.count()

15

In [21]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
|[T_AMC, T_GME]  |1   |
|[T_AMC, T_TSLA] |1   |
+----------------+----+



In [22]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+--------------------------------+----------+----------+------------------+---------------------+
|antecedent                      |consequent|confidence|lift              |support              |
+--------------------------------+----------+----------+------------------+---------------------+
|[T_MSFT, T_NVDA, T_TSLA]        |[T_AMC]   |0.25      |8.59375           |0.0036363636363636364|
|[T_MSFT, T_NVDA, T_TSLA, T_AAPL]|[T_AMC]   |0.25      |8.59375           |0.0036363636363636364|
|[T_AMC, T_TSLA, T_AAPL]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_AAPL]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA, T_AAPL] |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA]         |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA]                 |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA, T_A

**Initial parameters**:
- `minSupport = 0.001` (0.1%) – Very low threshold to capture even rare associations  
- `minConfidence = 0.1` (10%) – Low confidence to surface a broad range of potential patterns  

Analysis focused on a set of high-salience tickers:  
`["AAPL", "TSLA", "NVDA", "GME", "AMC", "MSFT"]`, which dominate discussion during the 2020–2022 period.

**Outcome**:  
The model identified frequent pairs and generated association rules across these assets.

**Challenge**:  
Simple substring matching may introduce false positives (e.g., "GME" matching substrings within longer words such as "SEGMENT"), though the high frequency of true ticker mentions in this dataset limits the overall impact of such cases.

**Adjustment**:
- `minSupport = 0.002` (0.2%) – Doubled the support threshold  
- `minConfidence = 0.3` (30%) – Increased confidence to emphasize stronger co-occurrence patterns  

**Test**:  
Whether tightening parameters reveals clearer and more interpretable market sentiment structures.

**Outcome**:  
Results remain largely similar, indicating that the most prominent associations are robust to threshold choice.


In [23]:
fp = FPGrowth(itemsCol="items", minSupport=0.002, minConfidence=0.3).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

15

In [24]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
|[T_AMC, T_GME]  |1   |
|[T_AMC, T_TSLA] |1   |
+----------------+----+



In [25]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+-------------------------------+----------+----------+------------------+---------------------+
|antecedent                     |consequent|confidence|lift              |support              |
+-------------------------------+----------+----------+------------------+---------------------+
|[T_AMC, T_TSLA, T_AAPL]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_AAPL]        |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_NVDA, T_TSLA, T_AAPL]|[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_TSLA]                |[T_MSFT]  |1.0       |2.669902912621359 |0.0036363636363636364|
|[T_AMC, T_MSFT, T_TSLA]        |[T_NVDA]  |1.0       |2.1484375         |0.0036363636363636364|
|[T_AMC, T_MSFT, T_TSLA, T_AAPL]|[T_NVDA]  |1.0       |2.1484375         |0.0036363636363636364|
|[T_AMC, T_TSLA]              

In [26]:
fp = FPGrowth(itemsCol="items", minSupport=0.005, minConfidence=0.2).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

13

In [27]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
+----------------+----+



In [28]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------+----------+-------------------+------------------+--------------------+
|antecedent              |consequent|confidence         |lift              |support             |
+------------------------+----------+-------------------+------------------+--------------------+
|[T_MSFT, T_NVDA, T_TSLA]|[T_AAPL]  |1.0                |1.4473684210526316|0.014545454545454545|
|[T_AMC, T_AAPL]         |[T_MSFT]  |0.5                |1.3349514563106795|0.007272727272727273|
|[T_GME, T_TSLA, T_AAPL] |[T_MSFT]  |0.5                |1.3349514563106795|0.007272727272727273|
|[T_GME, T_AAPL]         |[T_MSFT]  |0.45454545454545453|1.2135922330097086|0.01818181818181818 |
|[T_GME]                 |[T_MSFT]  |0.4375             |1.1680825242718447|0.05090909090909091 |
|[T_MSFT]                |[T_AAPL]  |0.7378640776699029 |1.0679611650485437|0.27636363636363637 |
|[T_AAPL]                |[T_MSFT]  |0.4                |1.0679611650485437|0.27636363636363637 |
|[T_AMC]            

**Further adjustment**:
- `minSupport = 0.01` (1%) – Highest support threshold tested, focusing only on the most frequently co-occurring ticker pairs  
- `minConfidence = 0.05` (5%) – Lower confidence threshold to retain all associations that meet the stricter support criterion  

**Test**:  
A high support threshold ensures that only consistently discussed ticker pairs are retained, while a low confidence threshold avoids prematurely filtering out associations that may be structurally important but not strongly directional.

**Outcome**:  
Results are slightly narrowed due to the higher support requirement, but the dominant ticker pairs remain largely similar, indicating robustness of core co-occurrence patterns.

In [29]:
fp = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.05).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
pairs.count()

13

In [30]:
pairs.show(20, truncate=False)

+----------------+----+
|pair            |freq|
+----------------+----+
|[T_AAPL, T_TSLA]|85  |
|[T_AAPL, T_MSFT]|76  |
|[T_AAPL, T_NVDA]|65  |
|[T_NVDA, T_TSLA]|61  |
|[T_MSFT, T_TSLA]|28  |
|[T_MSFT, T_NVDA]|22  |
|[T_GME, T_MSFT] |14  |
|[T_GME, T_TSLA] |12  |
|[T_AAPL, T_GME] |11  |
|[T_GME, T_NVDA] |7   |
|[T_AAPL, T_AMC] |4   |
|[T_AMC, T_NVDA] |3   |
|[T_AMC, T_MSFT] |3   |
+----------------+----+



In [31]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------+----------+-------------------+------------------+--------------------+
|antecedent              |consequent|confidence         |lift              |support             |
+------------------------+----------+-------------------+------------------+--------------------+
|[T_MSFT, T_NVDA, T_TSLA]|[T_AAPL]  |1.0                |1.4473684210526316|0.014545454545454545|
|[T_MSFT, T_TSLA]        |[T_GME]   |0.14285714285714285|1.2276785714285714|0.014545454545454545|
|[T_GME, T_AAPL]         |[T_MSFT]  |0.45454545454545453|1.2135922330097086|0.01818181818181818 |
|[T_MSFT]                |[T_GME]   |0.13592233009708737|1.1680825242718447|0.05090909090909091 |
|[T_GME]                 |[T_MSFT]  |0.4375             |1.1680825242718447|0.05090909090909091 |
|[T_MSFT]                |[T_AAPL]  |0.7378640776699029 |1.0679611650485437|0.27636363636363637 |
|[T_AAPL]                |[T_MSFT]  |0.4                |1.0679611650485437|0.27636363636363637 |
|[T_AMC]            

In [32]:
rules.orderBy("support").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+-------------------+
|          antecedent|consequent|             support|         confidence|
+--------------------+----------+--------------------+-------------------+
|             [T_AMC]|  [T_NVDA]| 0.01090909090909091|              0.375|
|             [T_AMC]|  [T_MSFT]| 0.01090909090909091|              0.375|
|[T_MSFT, T_NVDA, ...|  [T_TSLA]|0.014545454545454545|0.36363636363636365|
|     [T_GME, T_AAPL]|  [T_TSLA]|0.014545454545454545|0.36363636363636365|
|     [T_GME, T_TSLA]|  [T_AAPL]|0.014545454545454545| 0.3333333333333333|
|    [T_MSFT, T_TSLA]|   [T_GME]|0.014545454545454545|0.14285714285714285|
|     [T_GME, T_TSLA]|  [T_MSFT]|0.014545454545454545| 0.3333333333333333|
|[T_MSFT, T_NVDA, ...|  [T_AAPL]|0.014545454545454545|                1.0|
|             [T_AMC]|  [T_AAPL]|0.014545454545454545|                0.5|
|    [T_MSFT, T_TSLA]|  [T_NVDA]|0.014545454545454545|0.14285714285714285|
+--------------------+---

In [33]:
rules.orderBy("confidence").select("antecedent", "consequent", "support", "confidence").show(10)

+----------------+----------+--------------------+-------------------+
|      antecedent|consequent|             support|         confidence|
+----------------+----------+--------------------+-------------------+
|        [T_NVDA]|   [T_GME]|0.025454545454545455|          0.0546875|
|        [T_AAPL]|   [T_GME]|                0.04|0.05789473684210526|
|[T_NVDA, T_TSLA]|  [T_MSFT]|0.014545454545454545|0.06557377049180328|
|[T_MSFT, T_AAPL]|   [T_GME]| 0.01818181818181818|0.06578947368421052|
|        [T_TSLA]|   [T_GME]| 0.04363636363636364|0.08333333333333333|
|        [T_MSFT]|   [T_GME]| 0.05090909090909091|0.13592233009708737|
|[T_MSFT, T_TSLA]|   [T_GME]|0.014545454545454545|0.14285714285714285|
|[T_MSFT, T_TSLA]|  [T_NVDA]|0.014545454545454545|0.14285714285714285|
|[T_MSFT, T_AAPL]|  [T_NVDA]|                0.04|0.14473684210526316|
|[T_NVDA, T_AAPL]|  [T_MSFT]|                0.04|0.16923076923076924|
+----------------+----------+--------------------+-------------------+
only s

**Parameter choice**:
- `minSupport = 0.01` (1%) – Highest support threshold tested, focusing only on the most common co-occurrence patterns  
- `minConfidence = 0.05` (5%) – Very low confidence to retain all associations that meet the strict support requirement  

**Test**:  
Whether enforcing extremely high support while allowing low confidence reveals the *core* ticker groupings, patterns that appear frequently enough to be statistically meaningful regardless of predictive strength. This step also explicitly examines itemsets of size three or greater to capture more complex, multi-ticker discussion structures.

**Outcome**:  
Results are largely similar to the previous case, with a small number of stable higher-order itemsets emerging, suggesting that multi-ticker associations are consistent and robust within the 2020–2022 discussion data.

In [34]:
fp = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.05).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))
pairs = (freq
    .filter(F.size("items") >= 3)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
print(pairs.count())
pairs.show(20, truncate=False)

[Stage 156:>                                                        (0 + 4) / 4]

8
+--------------------------------+----+
|pair                            |freq|
+--------------------------------+----+
|[T_AAPL, T_MSFT, T_TSLA]        |19  |
|[T_AAPL, T_NVDA, T_TSLA]        |16  |
|[T_AAPL, T_MSFT, T_NVDA]        |11  |
|[T_AAPL, T_GME, T_MSFT]         |5   |
|[T_MSFT, T_NVDA, T_TSLA]        |4   |
|[T_AAPL, T_GME, T_TSLA]         |4   |
|[T_AAPL, T_MSFT, T_NVDA, T_TSLA]|4   |
|[T_GME, T_MSFT, T_TSLA]         |4   |
+--------------------------------+----+



**Test**: Looking at triplets (3+ tickers) to find complex co-mention patterns.

**Outcome**: Triplets do appear in this dataset, but the set of frequent 3+ itemsets remains limited because:
1. With only 6 tickers, we have a constrained search space (20 possible triplets)
2. The high support threshold (1%) requires 3+ ticker combinations to occur repeatedly across many submissions


### Expanded Ticker Set

In [16]:
from pyspark.sql import functions as F, types as T

comments = commentsDF.select(
    F.col("id").alias("id"),
    F.lit("comment").alias("src"),
    F.col("body").alias("text")
).where(F.col("text").isNotNull())

submissions = submissionsDF.select(
    F.col("id").alias("id"),
    F.lit("submission").alias("src"),
    F.concat_ws(" ", F.coalesce("title", F.lit("")), F.coalesce("selftext", F.lit(""))).alias("text")
).where(F.col("text").isNotNull())

In [17]:
comments = comments.cache()
submissions = submissions.cache()

25/12/12 06:32:32 WARN CacheManager: Asked to cache already cached data.
25/12/12 06:32:32 WARN CacheManager: Asked to cache already cached data.


In [18]:
from pyspark.sql import functions as F, types as T

tickers = ["AAPL", "AMZN", "MSFT", "GOOG", "META", "TXN", "ADP", "BSX", "APH", "ISRG", "GILD", "DE", "SYK", "ETN", "COF", "LLY", "UNH", "LOW", "HON", "PG", "ADI", "SCHW", "PLD", "CTRA", "KKR"]
ticker_set = set(tickers)

@F.udf(T.ArrayType(T.StringType()))
def extract_simple_tickers(text: str):
    if not text:
        return []
    s = text.upper()
    found = [t for t in ticker_set if t in s]
    return [f"T_{t}" for t in found]

In [19]:
submissions = submissions.select(
    "id",
    extract_simple_tickers(F.col("text")).alias("tickers")
)

In [20]:
comments = comments.select(
    "id",
    extract_simple_tickers("text").alias("tickers")
)

In [21]:
comment_groups = (comments
    .groupBy("id")
    .agg(F.array_distinct(F.flatten(F.collect_list("tickers"))).alias("c_tickers")))

In [22]:
baskets = (submissions.alias("s")
    .join(comment_groups.alias("c"), F.col("s.id") == F.col("c.id"), "left")
    .select(
        F.col("s.id"),
        F.array_distinct(F.array_union(F.coalesce(F.col("s.tickers"), F.array()),
                                       F.coalesce(F.col("c.c_tickers"), F.array()))).alias("items"))
    .filter(F.size("items") >= 2))

In [24]:
from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.1).fit(baskets)
freq = fp.freqItemsets.orderBy(F.desc("freq"))

In [25]:
pairs = (freq
    .filter(F.size("items") == 2)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))

In [26]:
pairs.show(20, truncate=False)

[Stage 20:===================================================>    (11 + 1) / 12]

+---------------+-----+
|pair           |freq |
+---------------+-----+
|[T_DE, T_LLY]  |33799|
|[T_DE, T_LOW]  |29455|
|[T_ADI, T_DE]  |20192|
|[T_LLY, T_LOW] |17292|
|[T_ADI, T_LLY] |11782|
|[T_ADI, T_LOW] |11388|
|[T_DE, T_HON]  |5419 |
|[T_DE, T_PG]   |4279 |
|[T_HON, T_LLY] |3582 |
|[T_DE, T_GOOG] |3348 |
|[T_HON, T_LOW] |2945 |
|[T_LLY, T_PG]  |2625 |
|[T_APH, T_DE]  |2576 |
|[T_LOW, T_PG]  |2515 |
|[T_ADI, T_HON] |2191 |
|[T_GOOG, T_LLY]|2156 |
|[T_GOOG, T_LOW]|1886 |
|[T_ADI, T_PG]  |1829 |
|[T_APH, T_LLY] |1788 |
|[T_DE, T_META] |1679 |
+---------------+-----+
only showing top 20 rows



In [27]:
rules = fp.associationRules.orderBy(F.desc("lift"))
rules.show(20, truncate=False)

+------------------------------------+----------+------------------+------------------+---------------------+
|antecedent                          |consequent|confidence        |lift              |support              |
+------------------------------------+----------+------------------+------------------+---------------------+
|[T_MSFT, T_AAPL, T_GOOG]            |[T_AMZN]  |0.7983870967741935|41.76351658337119 |0.0015679939180841965|
|[T_MSFT, T_AAPL, T_GOOG, T_DE]      |[T_AMZN]  |0.7909090909090909|41.37234314980794 |0.001377934049225506 |
|[T_AMZN, T_AAPL, T_GOOG]            |[T_MSFT]  |0.668918918918919 |34.90429975429976 |0.0015679939180841965|
|[T_AMZN, T_AAPL, T_GOOG, T_DE]      |[T_MSFT]  |0.6641221374045801|34.65400290202511 |0.001377934049225506 |
|[T_AAPL, T_GOOG, T_LOW, T_LLY, T_DE]|[T_AMZN]  |0.6238532110091743|32.633673601240474|0.0010770059235325794|
|[T_AAPL, T_GOOG, T_LOW, T_LLY]      |[T_AMZN]  |0.6181818181818182|32.33700384122919 |0.0010770059235325794|
|[T_AAPL, 

In [28]:
rules.orderBy("support").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+------------------+
|          antecedent|consequent|             support|        confidence|
+--------------------+----------+--------------------+------------------+
|[T_COF, T_HON, T_...|   [T_LLY]|0.001013652633913016|0.9846153846153847|
|      [T_ADP, T_HON]|    [T_DE]|0.001013652633913016|0.8533333333333334|
|[T_COF, T_HON, T_...|   [T_ADI]|0.001013652633913016|0.7710843373493976|
|[T_COF, T_HON, T_...|    [T_DE]|0.001013652633913016|               1.0|
|       [T_ADP, T_DE]|   [T_HON]|0.001013652633913016| 0.460431654676259|
|[T_COF, T_ADI, T_...|   [T_HON]|0.001013652633913016|0.4050632911392405|
|[T_ADP, T_LLY, T_DE]|   [T_ADI]|0.001013652633913016|0.6808510638297872|
|[T_COF, T_ADI, T_...|   [T_HON]|0.001013652633913016|0.4050632911392405|
|      [T_ADP, T_LOW]|   [T_ADI]|0.001013652633913016|0.7111111111111111|
|[T_COF, T_HON, T_...|   [T_LOW]|0.001013652633913016|0.9014084507042254|
+--------------------+----------+-----

In [29]:
rules.orderBy("confidence").select("antecedent", "consequent", "support", "confidence").show(10)

+--------------------+----------+--------------------+-------------------+
|          antecedent|consequent|             support|         confidence|
+--------------------+----------+--------------------+-------------------+
|             [T_HON]|  [T_GOOG]|0.009487155120529634|0.10026782725142283|
|[T_GOOG, T_ADI, T...|  [T_AMZN]|0.001362095726820...|0.10035005834305717|
|[T_APH, T_LOW, T_...|  [T_META]|0.002090658557445...|0.10038022813688213|
|[T_APH, T_LOW, T_...|  [T_META]|0.002074820235040...| 0.1004601226993865|
|[T_GOOG, T_ADI, T...|  [T_AMZN]|0.001362095726820...|0.10070257611241218|
|[T_GOOG, T_ADI, T...|  [T_AMZN]|0.001599670562893...|0.10140562248995984|
|[T_GOOG, T_ADI, T...|  [T_AMZN]|0.001583832240489...|0.10152284263959391|
|       [T_ADI, T_DE]|   [T_HON]| 0.03257942918686053|0.10187202852614897|
|[T_HON, T_LLY, T_DE]|   [T_APH]|0.005400867940067788|0.10224887556221889|
|             [T_COF]|  [T_GOOG]|0.001140359213152143|0.10449927431059507|
+--------------------+---

In [30]:
pairs = (freq
    .filter(F.size("items") >= 3)
    .withColumn("pair", F.array_sort("items"))
    .select("pair", "freq")
    .orderBy(F.desc("freq")))
print(pairs.count())
pairs.show(20, truncate=False)

589
+---------------------------+-----+
|pair                       |freq |
+---------------------------+-----+
|[T_DE, T_LLY, T_LOW]       |15991|
|[T_ADI, T_DE, T_LLY]       |10941|
|[T_ADI, T_DE, T_LOW]       |10482|
|[T_ADI, T_LLY, T_LOW]      |7582 |
|[T_ADI, T_DE, T_LLY, T_LOW]|7452 |
|[T_DE, T_HON, T_LLY]       |3335 |
|[T_DE, T_HON, T_LOW]       |2801 |
|[T_DE, T_LLY, T_PG]        |2533 |
|[T_DE, T_LOW, T_PG]        |2427 |
|[T_HON, T_LLY, T_LOW]      |2318 |
|[T_DE, T_HON, T_LLY, T_LOW]|2291 |
|[T_ADI, T_DE, T_HON]       |2057 |
|[T_DE, T_GOOG, T_LLY]      |2031 |
|[T_LLY, T_LOW, T_PG]       |1901 |
|[T_DE, T_LLY, T_LOW, T_PG] |1884 |
|[T_DE, T_GOOG, T_LOW]      |1788 |
|[T_ADI, T_DE, T_PG]        |1773 |
|[T_ADI, T_HON, T_LLY]      |1707 |
|[T_APH, T_DE, T_LLY]       |1700 |
|[T_ADI, T_DE, T_HON, T_LLY]|1683 |
+---------------------------+-----+
only showing top 20 rows



**Ticker frequencies**: If a stock is mentioned very frequently, it will naturally appear in many pairs - but that doesn't necessarily mean meaningful association.

In [31]:
tick_freq = (baskets
    .select(F.explode("items").alias("ticker"))
    .groupBy("ticker")
    .count()
    .orderBy(F.desc("count"))
)
tick_freq.show(15, truncate=False)

25/12/12 06:41:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:41:49 WARN RowBasedKeyValueBatch: Calling spill() on

+------+-----+
|ticker|count|
+------+-----+
|T_DE  |58672|
|T_LLY |36450|
|T_LOW |31988|
|T_ADI |22107|
|T_HON |5974 |
|T_PG  |4510 |
|T_GOOG|3681 |
|T_APH |2796 |
|T_META|1874 |
|T_AAPL|1558 |
|T_MSFT|1210 |
|T_AMZN|1207 |
|T_COF |689  |
|T_SCHW|624  |
|T_UNH |342  |
+------+-----+
only showing top 15 rows



**Pair Frequencies**: Building all possible ticker pairs manually using `combinations` and counting their co-occurrence frequency. The raw co-occurrence counts without FP-Growth's support/confidence filtering, helping us understand if the algorithm is appropriately filtering or if we're missing important patterns.

In [32]:
from itertools import combinations
from pyspark.sql.types import ArrayType, StringType

pairs_udf = F.udf(lambda xs: [tuple(sorted(x)) for x in combinations(set(xs), 2)],
                  ArrayType(ArrayType(StringType())))

pair_freq = (baskets
    .select(F.explode(pairs_udf("items")).alias("pair"))
    .select(F.col("pair")[0].alias("t1"), F.col("pair")[1].alias("t2"))
    .groupBy("t1","t2").count()
    .orderBy(F.desc("count"))
)
pair_freq.show(15, truncate=False)

25/12/12 06:44:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:44:23 WARN RowBasedKeyValueBatch: Calling spill() on

+-----+------+-----+
|t1   |t2    |count|
+-----+------+-----+
|T_DE |T_LLY |33799|
|T_DE |T_LOW |29455|
|T_ADI|T_DE  |20192|
|T_LLY|T_LOW |17292|
|T_ADI|T_LLY |11782|
|T_ADI|T_LOW |11388|
|T_DE |T_HON |5419 |
|T_DE |T_PG  |4279 |
|T_HON|T_LLY |3582 |
|T_DE |T_GOOG|3348 |
|T_HON|T_LOW |2945 |
|T_LLY|T_PG  |2625 |
|T_APH|T_DE  |2576 |
|T_LOW|T_PG  |2515 |
|T_ADI|T_HON |2191 |
+-----+------+-----+
only showing top 15 rows



## Correlation Analysis

Are the association rules just reflecting ticker popularity, or are they revealing genuine relationships?

**High correlation** = Rules mostly reflect popular stocks appearing together by chance

**Low correlation** = Rules capture genuine co-mention patterns beyond base popularity

If correlation is high, we need to normalize or use lift more heavily to find meaningful associations.

In [33]:
lhs_freq = tick_freq.select(F.col("ticker").alias("lhs"), F.col("count").alias("lhs_freq"))

rules_pop = (rules
    .where(F.size("antecedent")==1)
    .select(F.col("antecedent")[0].alias("lhs"), "support","confidence")
    .join(lhs_freq, "lhs", "left")
)

rules_pop.agg(
    F.corr("lhs_freq","support").alias("corr_support"),
    F.corr("lhs_freq","confidence").alias("corr_confidence")
).show()

25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:41 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/12/12 06:46:53 WARN RowBasedKeyValueBatch: Calling spill() on

+------------------+-------------------+
|      corr_support|    corr_confidence|
+------------------+-------------------+
|0.9285055793411963|0.11092350363776148|
+------------------+-------------------+



**Outcome**:

Very strong positive correlation - Rules involving more popular tickers also tend to have higher support — i.e., those tickers appear in a larger fraction of baskets, so their rules are more frequent. Popularity directly drives support.

Near-zero correlation - Popularity of a ticker has almost no effect on rule confidence — common tickers don’t necessarily make stronger conditional associations; they just occur more often.

## Experimentation Summary

Testing multiple combinations of `minSupport` and `minConfidence` clarified the trade-off between pattern abundance and interpretability. Lower thresholds surfaced many associations driven by overall ticker frequency, while higher support thresholds isolated a smaller set of stable, frequently co-occurring patterns.

Expanding the ticker set from 6 to 25 substantially increased the richness of the itemset space, enabling more nuanced co-mention structures to emerge beyond dominant large-cap stocks. This expansion reduced over-concentration on a few assets and provided a clearer view of comparative and thematic discussion patterns.

Manually constructing ticker pairs and correlating rule metrics with baseline ticker frequency helped validate that FP-Growth results were not purely artifacts of popularity. While support was strongly correlated with ticker frequency, confidence showed little dependence on popularity, indicating that frequent mentions do not necessarily translate into stronger conditional associations.

Exploring higher-order itemsets (3+ tickers) yielded relatively sparse results, reflecting both the limited combinatorial space and the structure of Reddit discussions, which tend to focus on single stocks or direct comparisons between two assets rather than larger coordinated groups.

Finally, the analysis highlighted a methodological limitation in substring-based ticker matching, which can introduce false positives (e.g., "DE" matching "DELETED"). More robust approaches—such as enforcing word boundaries or using named-entity recognition—would improve precision in future iterations.
